# Imports and Initialisation

In [2]:
import boto3
from datetime import datetime, timezone

# Initialize Batch client
batch = boto3.client('batch')

# Constants
START_DATE = datetime(2026, 3, 27, tzinfo=timezone.utc)  # 27 March 2026
JOB_QUEUES = ['prod-xlsx-to-pdf-queue']  # Replace with actual job queue(s)
JOB_STATUS = 'FAILED'

# Functions

In [3]:
def get_failed_jobs(job_queue, start_time):
    """
    Fetch failed jobs since a given start time from a job queue.
    """
    failed_jobs = []
    next_token = None

    while True:
        response = batch.list_jobs(
            jobQueue=job_queue,
            jobStatus=JOB_STATUS,
            nextToken=next_token if next_token else '',
            maxResults=100
        )

        for job in response['jobSummaryList']:
            created_at = datetime.fromtimestamp(job['createdAt'] / 1000, tz=timezone.utc)
            if created_at >= start_time:
                failed_jobs.append(job['jobId'])

        next_token = response.get('nextToken')
        if not next_token:
            break

    return failed_jobs

def rerun_job(job_id):
    """
    Re-run a job using the same job definition, queue, and overrides.
    """
    job_detail = batch.describe_jobs(jobs=[job_id])['jobs'][0]

    # Prepare containerOverrides as a dict
    container_overrides = {}
    if 'container' in job_detail and 'environment' in job_detail['container']:
        container_overrides['environment'] = job_detail['container']['environment']

    # Prepare retryStrategy with attempts
    retry_strategy = job_detail.get('retryStrategy', {})
    if 'attempts' not in retry_strategy:
        retry_strategy['attempts'] = 1  # Default to 1 attempt if not present

    new_job = batch.submit_job(
        jobName=job_detail['jobName'] + '-retry',
        jobQueue=job_detail['jobQueue'],
        jobDefinition=job_detail['jobDefinition'],
        containerOverrides=container_overrides,
        retryStrategy=retry_strategy
    )
    print(f"Re-submitted job: {new_job['jobId']} (original: {job_id})")


# Main Function

In [4]:
def main():
    for queue in JOB_QUEUES:
        print(f"Checking failed jobs in queue: {queue}")
        failed_jobs = get_failed_jobs(queue, START_DATE)

        print(f"Found {len(failed_jobs)} failed jobs since {START_DATE.date()}")

        for job_id in failed_jobs:
            try:
                rerun_job(job_id)
            except Exception as e:
                print(f"Failed to rerun job {job_id}: {e}")

if __name__ == '__main__':
    main()

Checking failed jobs in queue: prod-xlsx-to-pdf-queue
Found 1284 failed jobs since 2026-03-27
Re-submitted job: abe42a07-4951-45a9-b562-302a6936d914 (original: fb2dda86-50bc-4ef4-83c2-13b80a4b706d)
Re-submitted job: 0aba9e1d-8503-4ed4-b904-6905d2fbaf99 (original: 6d13ae8f-1b00-43f4-b266-f269956430ec)
Re-submitted job: 4aee38d5-3f4c-4134-9031-40e2cf41fb20 (original: a679f44a-b64a-477f-842e-3ee960f92544)
Re-submitted job: 2179335d-a986-4401-b5b2-5ad0c9470377 (original: 796ce9fb-a4a0-464b-a925-a91e2debb3b6)
Re-submitted job: c347f00c-e53e-4726-95b6-73287d4b90d4 (original: bb9aa29e-2952-480a-b9b1-404027b7e5cc)
Re-submitted job: 3d4527ff-6bea-4969-9e5d-04a7d765dea9 (original: e2a2a9e0-bae7-4eef-965f-3ef906a4ff71)
Re-submitted job: 7639ad26-b6d7-4a85-9776-b28c5c7d16b2 (original: 6c32558a-5ea8-4b00-9f86-c15ab008aeff)
Re-submitted job: d7e88c7e-39a4-4a99-8dfc-748f91f2f14a (original: f2b3ac71-e0c7-45f7-925e-9d18bcba4310)
Re-submitted job: e8d5f1ad-bd30-4418-9575-1297e9340569 (original: 15bbdd52